In [21]:
import subprocess
import os
import csv
import pandas as pd
from bayes_opt import BayesianOptimization
from pathlib import Path
import numpy as np
from simulation_functions import call_vot_simulation, intialise_java, call_stt_simulation,STT_INIT_POINTS, STT_N_ITER
from scipy import stats

VOT config

In [22]:
#VOT config
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
vot_pbounds = {'alphaWalk' :(0,0) ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : (0,7),
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': default_beta_bounds,
           'betaCostLow':(-0.4,-0.2) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.2,-0.01),
           'weightWalk': (1,2),
           'weightWait': (1,2),
           'weightFeeder': (1,2), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

STT config

In [23]:
stt_pbounds = {
    'alphaWalk' :(0,0) ,
    'alphaBike' : defaultAlpha_bounds,
    'alphaCarDriver' : (0,7),
    'alphaCarPassenger' : defaultAlpha_bounds,
    'alphaBus' : defaultAlpha_bounds,
    'alphaTrain' : defaultAlpha_bounds,
    'betaTimeWalk' : (-0.04,-0.04),
    'betaTimeBike' : (-0.03,-0.03),
    'betaTimeCarDriver': (-0.02,-0.02),
    'betaTimeCarPassenger' : (-0.02,-0.02),
    'betaTimeBus' : (-0.02,-0.02),
    'betaTimeTrain' : (-0.02,-0.02),
    'betaCostCarDriver': (-0.15,-0.15),
    'betaCostCarPassenger' : (-0.15,-0.15),
    'betaCostBus': (-0.1,-0.1),
    'betaCostTrain': (-0.1,-0.1),
    'betaTimeWalkTransport': (-0.03,-0.03),
    'betaChangesTransport': (-0.3,-0.3)}

In [24]:
intialise_java()

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



In [25]:

rq_1_config = {
    "config_file": "src/main/resources/config_DHZW_rq1-1.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq1-1.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq1-1.csv',
    "other_args" : "--use_random_seed true"
}
rq1_1_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_1_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq1_1_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -30.32145 | 0.0       | 2.2032449 | 0.0008006 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -12.56200 | 0.0       | -3.018985 | 5.6052119 | 4.6826157 | -1.865758 | 1.9232261 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 3         | -23.28914 | 0.0   

In [26]:
print(rq1_1_optimiser.max)

{'target': np.float64(-7.7043484545040695), 'params': {'alphaWalk': np.float64(0.0), 'alphaBike': np.float64(-1.424587850263955), 'alphaCarDriver': np.float64(7.0), 'alphaCarPassenger': np.float64(3.7663075687683563), 'alphaBus': np.float64(-0.32797988456826377), 'alphaTrain': np.float64(-0.41980997155216987), 'betaTimeWalk': np.float64(-0.04), 'betaTimeBike': np.float64(-0.03), 'betaTimeCarDriver': np.float64(-0.02), 'betaTimeCarPassenger': np.float64(-0.02), 'betaTimeBus': np.float64(-0.02), 'betaTimeTrain': np.float64(-0.02), 'betaCostCarDriver': np.float64(-0.15), 'betaCostCarPassenger': np.float64(-0.15), 'betaCostBus': np.float64(-0.1), 'betaCostTrain': np.float64(-0.1), 'betaTimeWalkTransport': np.float64(-0.03), 'betaChangesTransport': np.float64(-0.3)}}


In [27]:
rq_1_config_no_seed = rq_1_config
rq_1_config_no_seed["other_args"] = "--use_random_seed false"
base_pop_results = []
for i in range(0,30):
    base_pop_results.append(- call_stt_simulation(config=rq_1_config_no_seed,**rq1_1_optimiser.max['params']))


In [28]:
base_pop_results

[7.707496601071758,
 7.707436184480112,
 7.7073593723417755,
 7.708039607423291,
 7.705864289765003,
 7.709259901004068,
 7.707693312338857,
 7.70593735268445,
 7.706104494273714,
 7.704753692947246,
 7.7058294949652275,
 7.706207809803259,
 7.708250154429711,
 7.706808264388042,
 7.7076438703080825,
 7.7053095095734045,
 7.705551949630614,
 7.70548003397435,
 7.706312792905369,
 7.708235133983464,
 7.707889724324994,
 7.706785790604341,
 7.705627272353155,
 7.7076598846156745,
 7.706996388961764,
 7.708652864074839,
 7.708328253582076,
 7.708545033254906,
 7.705698322026233,
 7.705627426517699]

In [38]:
np.mean(base_pop_results)

np.float64(7.706912826086915)

In [39]:
np.std(base_pop_results)

np.float64(0.0011855677234802742)

In [40]:
rq1_1_optimiser.max['params']

{'alphaWalk': np.float64(0.0),
 'alphaBike': np.float64(-1.424587850263955),
 'alphaCarDriver': np.float64(7.0),
 'alphaCarPassenger': np.float64(3.7663075687683563),
 'alphaBus': np.float64(-0.32797988456826377),
 'alphaTrain': np.float64(-0.41980997155216987),
 'betaTimeWalk': np.float64(-0.04),
 'betaTimeBike': np.float64(-0.03),
 'betaTimeCarDriver': np.float64(-0.02),
 'betaTimeCarPassenger': np.float64(-0.02),
 'betaTimeBus': np.float64(-0.02),
 'betaTimeTrain': np.float64(-0.02),
 'betaCostCarDriver': np.float64(-0.15),
 'betaCostCarPassenger': np.float64(-0.15),
 'betaCostBus': np.float64(-0.1),
 'betaCostTrain': np.float64(-0.1),
 'betaTimeWalkTransport': np.float64(-0.03),
 'betaChangesTransport': np.float64(-0.3)}

RQ1.2

In [30]:

rq_2_config = {
    "config_file": "src/main/resources/config_DHZW_rq1-2.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq1-2.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq1-2.csv',
    "other_args" : "--use_random_seed true"
}
rq1_2_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_2_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq1_2_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER,
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -29.45069 | 0.0       | 2.2032449 | 0.0008006 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -8.093385 | 0.0       | -3.018985 | 5.6052119 | 4.6826157 | -1.865758 | 1.9232261 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 3         | -21.89966 | 0.0   

In [31]:
rq_2_config_no_seed = rq_2_config
rq_2_config_no_seed["other_args"] = "--use_random_seed false"
gen_synth_pop_results = []
for i in range(0,30):
    gen_synth_pop_results.append(- call_stt_simulation(config=rq_2_config_no_seed,**rq1_2_optimiser.max['params']))

In [32]:
gen_synth_pop_results

[0.9513533794692444,
 0.9540499640034996,
 0.9532166953980288,
 0.9513134915454037,
 0.9512552091672533,
 0.9569671989366763,
 0.9528637863647696,
 0.9504651377198182,
 0.955995576349999,
 0.9522752271417112,
 0.9551395816499839,
 0.9553854591161569,
 0.9523419406392566,
 0.9552537633758673,
 0.9517056127575784,
 0.9519883920606923,
 0.9539855575549248,
 0.9522222328483616,
 0.9553597806303266,
 0.9536908172784654,
 0.9513282607052767,
 0.954356151933595,
 0.9528620778037029,
 0.9563416282312993,
 0.9496571492099861,
 0.9537171992317458,
 0.9541091332242164,
 0.9531096367142212,
 0.9565184545949811,
 0.9515004144918009]

In [37]:
np.mean(gen_synth_pop_results)

np.float64(0.9533442970049614)

In [36]:
np.std(gen_synth_pop_results)

np.float64(0.0018916929225981218)

In [41]:
rq1_2_optimiser.max['params']

{'alphaWalk': np.float64(0.0),
 'alphaBike': np.float64(-1.6743398263290983),
 'alphaCarDriver': np.float64(6.848652656271703),
 'alphaCarPassenger': np.float64(4.447247691823898),
 'alphaBus': np.float64(-1.0383988136171385),
 'alphaTrain': np.float64(-2.812728063686222),
 'betaTimeWalk': np.float64(-0.04),
 'betaTimeBike': np.float64(-0.03),
 'betaTimeCarDriver': np.float64(-0.02),
 'betaTimeCarPassenger': np.float64(-0.02),
 'betaTimeBus': np.float64(-0.02),
 'betaTimeTrain': np.float64(-0.02),
 'betaCostCarDriver': np.float64(-0.15),
 'betaCostCarPassenger': np.float64(-0.15),
 'betaCostBus': np.float64(-0.1),
 'betaCostTrain': np.float64(-0.1),
 'betaTimeWalkTransport': np.float64(-0.03),
 'betaChangesTransport': np.float64(-0.3)}

Significance

In [34]:
baseline = np.array(base_pop_results)
improved = np.array(gen_synth_pop_results)

t_stat, p_value = stats.ttest_ind(baseline, improved)

print(f"P-value: {p_value}")
if p_value < 0.05:
    print("Statistically significant difference.")

P-value: 7.33223911792528e-195
Statistically significant difference.


In [35]:
#Confidence intervals
n1, n2 = len(baseline), len(improved)
mean1, mean2 = np.mean(baseline), np.mean(improved)
diff = mean1 - mean2


se_diff = np.sqrt(np.var(baseline, ddof=1)/n1 + np.var(improved, ddof=1)/n2)

df = n1 + n2 - 2
t_crit = stats.t.ppf(0.975, df)

lower_bound = diff - (t_crit * se_diff)
upper_bound = diff + (t_crit * se_diff)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"95% CI of the difference: [{lower_bound:.4f}, {upper_bound:.4f}]")

t-statistic: 16290.7138
p-value: 0.0000
95% CI of the difference: [6.7527, 6.7544]
